In [ ]:
!git clone https://github.com/cszn/KAIR.git
%cd KAIR

Cloning into 'KAIR'...
remote: Enumerating objects: 1804, done.
remote: Counting objects: 100% (690/690), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 1804 (delta 603), reused 567 (delta 567), pack-reused 1114 (from 1)
Receiving objects: 100% (1804/1804), 19.38 MiB | 43.71 MiB/s, done.
Resolving deltas: 100% (1084/1084), done.
/content/KAIR


In [ ]:
!pip install -r requirement.txt
!pip install einops timm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.6/77.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.4/299.4 kB 16.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_PATH = '/content/drive/MyDrive/Building-an-Adaptive-License-Plate-Recognition-and-Restoration-System-using-Multi-Task-Deep-Learning/Datasets/Datasets/module3'

TRAIN_GT = f'{BASE_PATH}/train/gt'
TRAIN_LQ = f'{BASE_PATH}/train/lq'
VAL_GT = f'{BASE_PATH}/val/gt'
VAL_LQ = f'{BASE_PATH}/val/lq'

for path, name in [(TRAIN_GT, 'Train GT'), (TRAIN_LQ, 'Train LQ'),
                    (VAL_GT, 'Val GT'), (VAL_LQ, 'Val LQ')]:
    count = len(os.listdir(path)) if os.path.exists(path) else 0
    print(f"{name}: {count} images")

Mounted at /content/drive
Train GT: 477 images
Train LQ: 477 images
Val GT: 177 images
Val LQ: 177 images


In [ ]:
import json

config = {
    "task": "real_sr",
    "scale": 1,
    "model": "plain",
    "gpu_ids": [0],

    "path": {
        "root": "/content/KAIR",
        "pretrained_netG": None,
        "models": "/content/KAIR/superresolution/swinir_license_plate",
        "log": "/content/KAIR/superresolution/swinir_license_plate",
        "options": "/content/KAIR/options/train_swinir_license.json"
    },

    "datasets": {
        "train": {
            "name": "license_train",
            "dataset_type": "paired",
            "dataroot_H": TRAIN_GT,
            "dataroot_L": TRAIN_LQ,
            "H_size": 128,
            "dataloader_shuffle": True,
            "dataloader_num_workers": 2,
            "dataloader_batch_size": 8
        },
        "test": {
            "name": "license_val",
            "dataset_type": "paired",
            "dataroot_H": VAL_GT,
            "dataroot_L": VAL_LQ
        }
    },

    "netG": {
        "net_type": "swinir",
        "upscale": 1,
        "in_chans": 3,
        "img_size": 128,
        "window_size": 8,
        "img_range": 1.0,
        "depths": [6, 6, 6, 6],
        "embed_dim": 60,
        "num_heads": [6, 6, 6, 6],
        "mlp_ratio": 2,
        "upsampler": "",
        "resi_connection": "1conv"
    },

    "train": {
        "G_lossfn_type": "l1",
        "G_lossfn_weight": 1.0,

        "G_optimizer_type": "adam",
        "G_optimizer_lr": 0.0002,
        "G_optimizer_wd": 0,
        "G_optimizer_betas": [0.9, 0.999],

        "G_scheduler_type": "MultiStepLR",
        "G_scheduler_milestones": [250000, 400000, 450000, 475000],
        "G_scheduler_gamma": 0.5,

        "G_regularizer_orthstep": None,
        "G_regularizer_clipstep": None,

        "checkpoint_test": 2500,
        "checkpoint_save": 2500,
        "checkpoint_print": 100
    },

    "netE": {
        "net_type": "swinir"
    }
}

os.makedirs('options', exist_ok=True)
with open('options/train_swinir_license.json', 'w') as f:
    json.dump(config, f, indent=2)

print("✓ Configuration saved to: options/train_swinir_license.json")

✓ Configuration saved to: options/train_swinir_license.json


In [ ]:
dataset_code = '''
import os
import random
import numpy as np
import cv2
import torch
import torch.utils.data as data
import utils.utils_image as util

class DatasetPaired(data.Dataset):
    """
    Dataset for paired HR (clean) and LR (damaged) images
    Handles variable-sized license plate images
    """
    def __init__(self, opt):
        super(DatasetPaired, self).__init__()
        self.opt = opt
        self.n_channels = opt.get('n_channels', 3)
        self.patch_size = opt.get('H_size', 128)

        # Get image paths
        self.paths_H = util.get_image_paths(opt['dataroot_H'])
        self.paths_L = util.get_image_paths(opt['dataroot_L'])

        # Match filenames
        self.paths_H = sorted(self.paths_H)
        self.paths_L = sorted(self.paths_L)

        assert len(self.paths_H) == len(self.paths_L), 'HR and LR datasets have different number of images'

    def _resize_image(self, img, size):
        """Resize image to fixed size while maintaining aspect ratio with padding"""
        h, w = img.shape[:2]

        # Calculate scaling factor to fit within size x size
        scale = min(size / h, size / w)
        new_h, new_w = int(h * scale), int(w * scale)

        # Resize image
        img_resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

        # Create padded image
        img_padded = np.zeros((size, size, 3), dtype=img.dtype)

        # Center the image
        y_offset = (size - new_h) // 2
        x_offset = (size - new_w) // 2
        img_padded[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = img_resized

        return img_padded

    def __getitem__(self, index):
        # Read images
        H_path = self.paths_H[index]
        L_path = self.paths_L[index]

        img_H = util.imread_uint(H_path, self.n_channels)
        img_L = util.imread_uint(L_path, self.n_channels)

        # Ensure same size
        h_min = min(img_H.shape[0], img_L.shape[0])
        w_min = min(img_H.shape[1], img_L.shape[1])
        img_H = img_H[:h_min, :w_min, :]
        img_L = img_L[:h_min, :w_min, :]

        # Resize to fixed size for training
        if self.opt['phase'] == 'train':
            img_H = self._resize_image(img_H, self.patch_size)
            img_L = self._resize_image(img_L, self.patch_size)

            # Random horizontal flip
            if random.random() < 0.5:
                img_L = np.fliplr(img_L)
                img_H = np.fliplr(img_H)

            # Random brightness adjustment (simulate different lighting)
            if random.random() < 0.3:
                factor = random.uniform(0.8, 1.2)
                img_L = np.clip(img_L * factor, 0, 255).astype(np.uint8)
        else:
            # For validation, also resize to consistent size
            img_H = self._resize_image(img_H, self.patch_size)
            img_L = self._resize_image(img_L, self.patch_size)

        # numpy to tensor
        img_H = util.uint2tensor3(img_H)
        img_L = util.uint2tensor3(img_L)

        return {'L': img_L, 'H': img_H, 'L_path': L_path, 'H_path': H_path}

    def __len__(self):
        return len(self.paths_H)
'''

with open('data/dataset_paired.py', 'w') as f:
    f.write(dataset_code)

print("Custom paired dataset class created")

Custom paired dataset class created


In [ ]:
training_script = '''
import os
import json
import torch
import logging
import numpy as np
from collections import OrderedDict
from utils import utils_logger
from utils import utils_image as util
from data.dataset_paired import DatasetPaired
from models.network_swinir import SwinIR as net

def main(json_path="options/train_swinir_license.json"):
    """
    Main training function for SwinIR license plate restoration
    """
    # Load options
    with open(json_path, 'r') as f:
        opt = json.load(f)

    opt['dist'] = False

    # Create directories
    os.makedirs(opt['path']['models'], exist_ok=True)
    os.makedirs(opt['path']['log'], exist_ok=True)

    # Setup logger
    logger_name = 'train'
    utils_logger.logger_info(logger_name, os.path.join(opt['path']['log'], 'train.log'))
    logger = logging.getLogger(logger_name)
    logger.info(json.dumps(opt, indent=2))

    # Seed
    torch.manual_seed(0)

    # Create datasets
    train_opt = opt['datasets']['train']
    train_opt['phase'] = 'train'
    train_set = DatasetPaired(train_opt)
    train_loader = torch.utils.data.DataLoader(
        train_set,
        batch_size=train_opt['dataloader_batch_size'],
        shuffle=train_opt['dataloader_shuffle'],
        num_workers=train_opt['dataloader_num_workers'],
        drop_last=True,
        pin_memory=True
    )

    test_opt = opt['datasets']['test']
    test_opt['phase'] = 'test'
    test_set = DatasetPaired(test_opt)
    test_loader = torch.utils.data.DataLoader(
        test_set, batch_size=1, shuffle=False,
        num_workers=1, drop_last=False, pin_memory=True
    )

    logger.info(f'Training samples: {len(train_set)}')
    logger.info(f'Validation samples: {len(test_set)}')

    # Create model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = net(
        upscale=opt['netG']['upscale'],
        in_chans=opt['netG']['in_chans'],
        img_size=opt['netG']['img_size'],
        window_size=opt['netG']['window_size'],
        img_range=opt['netG']['img_range'],
        depths=opt['netG']['depths'],
        embed_dim=opt['netG']['embed_dim'],
        num_heads=opt['netG']['num_heads'],
        mlp_ratio=opt['netG']['mlp_ratio'],
        upsampler=opt['netG']['upsampler'],
        resi_connection=opt['netG']['resi_connection']
    ).to(device)

    # Optimizer
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=opt['train']['G_optimizer_lr'],
        betas=opt['train']['G_optimizer_betas'],
        weight_decay=opt['train']['G_optimizer_wd']
    )

    # Scheduler
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=opt['train']['G_scheduler_milestones'],
        gamma=opt['train']['G_scheduler_gamma']
    )

    # Loss function
    criterion = torch.nn.L1Loss().to(device)

    # Training loop
    current_step = 0
    best_psnr = 0

    logger.info('Starting training...')

    for epoch in range(85):
        model.train()

        for i, train_data in enumerate(train_loader):
            current_step += 1

            # Data to device
            img_L = train_data['L'].to(device)
            img_H = train_data['H'].to(device)

            # Forward
            optimizer.zero_grad()
            img_E = model(img_L)
            loss = criterion(img_E, img_H)

            # Backward
            loss.backward()
            optimizer.step()

            # Update learning rate
            scheduler.step()

            # Logging
            if current_step % opt['train']['checkpoint_print'] == 0:
                lr = optimizer.param_groups[0]['lr']
                logger.info(f'Epoch: {epoch:3d} | Iter: {current_step:6d} | Loss: {loss.item():.4e} | LR: {lr:.3e}')

            # Validation
            if current_step % opt['train']['checkpoint_test'] == 0:
                model.eval()
                psnr_list = []

                with torch.no_grad():
                    for test_data in test_loader:
                        img_L = test_data['L'].to(device)
                        img_H = test_data['H'].to(device)

                        img_E = model(img_L)

                        # Calculate PSNR
                        img_E_np = util.tensor2uint(img_E)
                        img_H_np = util.tensor2uint(img_H)
                        psnr = util.calculate_psnr(img_E_np, img_H_np, border=0)
                        psnr_list.append(psnr)

                avg_psnr = np.mean(psnr_list)
                logger.info(f'Validation PSNR: {avg_psnr:.2f} dB')

                # Save best model
                if avg_psnr > best_psnr:
                    best_psnr = avg_psnr
                    torch.save({
                        'epoch': epoch,
                        'step': current_step,
                        'state_dict': model.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'psnr': best_psnr
                    }, os.path.join(opt['path']['models'], 'best_model.pth'))
                    logger.info(f'Best model saved! PSNR: {best_psnr:.2f} dB')

                model.train()

            # Save checkpoint
            if current_step % opt['train']['checkpoint_save'] == 0:
                torch.save({
                    'epoch': epoch,
                    'step': current_step,
                    'state_dict': model.state_dict(),
                    'optimizer': optimizer.state_dict()
                }, os.path.join(opt['path']['models'], f'model_{current_step}.pth'))
                logger.info(f'Checkpoint saved at step {current_step}')

            # Stop condition
            if current_step >= 500000:
                logger.info('Training completed!')
                return

if __name__ == '__main__':
    main()
'''

with open('train_license_plate.py', 'w') as f:
    f.write(training_script)

print("Training script created")

Training script created


In [ ]:
inference_script = '''
import os
import glob
import torch
import numpy as np
from utils import utils_image as util
from models.network_swinir import SwinIR as net

def inference(model_path, input_folder, output_folder):
    """
    Run inference on damaged license plates
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Load model
    model = net(
        upscale=1, in_chans=3, img_size=128, window_size=8,
        img_range=1., depths=[6, 6, 6, 6], embed_dim=60,
        num_heads=[6, 6, 6, 6], mlp_ratio=2, upsampler='',
        resi_connection='1conv'
    ).to(device)

    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['state_dict'], strict=True)
    model.eval()

    print(f"Model loaded from: {model_path}")
    if 'psnr' in checkpoint:
        print(f"Model PSNR: {checkpoint['psnr']:.2f} dB")

    # Create output folder
    os.makedirs(output_folder, exist_ok=True)

    # Get all images
    image_paths = sorted(glob.glob(os.path.join(input_folder, '*')))

    print(f"Found {len(image_paths)} images to process")

    with torch.no_grad():
        for idx, img_path in enumerate(image_paths):
            # Read image
            img_L = util.imread_uint(img_path, n_channels=3)
            img_L = util.uint2tensor4(img_L).to(device)

            # Inference
            img_E = model(img_L)

            # Save result
            img_E = util.tensor2uint(img_E)
            filename = os.path.basename(img_path)
            util.imsave(img_E, os.path.join(output_folder, filename))

            if (idx + 1) % 10 == 0:
                print(f"Processed {idx + 1}/{len(image_paths)} images")

    print(f"\\nAll results saved to: {output_folder}")

# Example usage:
if __name__ == "__main__":
    inference(
        model_path="superresolution/swinir_license_plate/best_model.pth",
        input_folder="/path/to/damaged/images",
        output_folder="results/restored"
    )
'''

with open('inference_license_plate.py', 'w') as f:
    f.write(inference_script)

print("Inference script created")


Inference script created


In [ ]:
!python train_license_plate.py

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
LogHandlers setup!
25-11-11 09:26:55.105 : {
  "task": "real_sr",
  "scale": 1,
  "model": "plain",
  "gpu_ids": [
    0
  ],
  "path": {
    "root": "/content/KAIR",
    "pretrained_netG": null,
    "models": "/content/KAIR/superresolution/swinir_license_plate",
    "log": "/content/KAIR/superresolution/swinir_license_plate",
    "options": "/content/KAIR/options/train_swinir_license.json"
  },
  "datasets": {
    "train": {
      "name": "license_train",
      "dataset_type": "paired",
      "dataroot_H": "/content/drive/MyDrive/Building-an-Adaptive-License-Plate-Recognition-and-Restoration-System-using-Multi-Task-Deep-Learning/Datasets/Datasets/module3/train/gt",
      "dataroot_L": "/content/drive/MyDrive/Buildi